In [1]:
# Save temperature on pressure fields

In [2]:
import os
import re
import cftime
import warnings
import xarray as xr
from collections import defaultdict
from utils.utils import get_scenario_config, load_file_list

In [6]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "hist"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/temp/file_paths/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/temp/temp_pres/"

In [7]:
def load_file(f):
    if not os.path.exists(f):
        raise ValueError(f"Missing: {f}")

    # Find variable
    pattern = re.compile(r"cam\.h0\.([^.]+)\.")
    m = pattern.search(f)
    varname = m.group(1)

    ds = xr.open_dataset(f)
    da = ds[varname]
    return da


# CESM2 naming convention shifts months by 1
# the data represents 2015-01 - 2020-12 but time coord shows 2015-02 - 2021-01
def minus_one_month(date):
    """Subtract one month from a cftime.DatetimeNoLeap object."""
    year, month = date.year, date.month
    if month == 1:
        return cftime.DatetimeNoLeap(year - 1, 12, date.day,
                                     date.hour, date.minute, date.second,
                                     date.microsecond, has_year_zero=date.has_year_zero)
    else:
        return cftime.DatetimeNoLeap(year, month - 1, date.day,
                                     date.hour, date.minute, date.second,
                                     date.microsecond, has_year_zero=date.has_year_zero)

In [8]:
for ens_num in ensemble_members:
    print(f"Processing {scenario}, ensemble {ens_num:02d}")
    # Load all file lists
    file_list = load_file_list(FILE_DIR, f"file_list_T_{scenario}_{ens_num:02d}.json")

    groups = defaultdict(list)
    for f in file_list:
        # Load variable
        da = load_file(f)
        groups[da.name].append(da)

    combined = {}
    for vbl, das in groups.items():
        combined[vbl] = xr.concat(
            das,
            dim="time",
            join="outer",
            combine_attrs="drop_conflicts"
        )

    # Get unit attribute from combined
    first_key = next(iter(combined))
    units = combined[first_key].attrs["units"]

    total_var = sum(combined.values())
    total_var.attrs["units"] = units

    # If the first month is February (2) then apply month fixer
    first_month = total_var.time.dt.month[0]
    if first_month == 2:
        print("Adjusting month indexing")
        new_time = [minus_one_month(t) for t in total_var["time"].values]
        total_var = total_var.assign_coords(time=new_time)
    # If the first month is January (1) don't apply month fixer
    elif first_month == 1:
        print("No month index adjusting needed")
    else:
        warnings.warn(f"First month: {first_month}, check dates in file")

    first_year = total_var.time.dt.year[0].item()
    last_year = total_var.time.dt.year[-1].item()

    out_file = f"T_{model}_{scenario}_{ens_num:02d}_{first_year}-{last_year}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    total_var.to_netcdf(out_path)

print("All processing complete.")

Processing hist, ensemble 01
Adjusting month indexing
All processing complete.
